# Generate images from text prompts with Stable Diffusion

Generates one image per prompt in `data/prompts.txt` with two models, Stable Diffusion v1.5 and v2.1, and saves them to `generated_images/v1-5/` and `generated_images/v2-1/`. File names follow the pattern `01_<prompt>.png`, which the embedding scripts rely on.

**Requirements:** a CUDA GPU (e.g. a free Google Colab GPU runtime) and a Hugging Face connection to download the models.

**Next step:** after generating the images, run `scripts/generated_images_embedding_extraction.py` to compute their embeddings.

# Install and import libraries

In [ ]:
!pip install diffusers transformers accelerate

In [ ]:
import gc
import os

import torch
from diffusers import StableDiffusionPipeline

# Settings

In [ ]:
# If you run this notebook on Google Colab, mount Google Drive (or upload the repository)
# and point the paths below to its location, for example:
# from google.colab import drive
# drive.mount('/content/drive')

PROMPT_PATH = "data/prompts.txt"
OUTPUT_ROOT = "generated_images"

# Output sub-folder -> Hugging Face model id
MODELS = {
    "v1-5": "sd-legacy/stable-diffusion-v1-5",  # community mirror: the original runwayml repo was removed
    "v2-1": "sd2-community/stable-diffusion-2-1",  # community mirror: the original stabilityai repo is deprecated
}

# Set an integer to make generation repeatable (same prompt -> same image).
# The committed images were generated without a fixed seed.
SEED = None

# Load prompts

In [ ]:
with open(PROMPT_PATH, "r") as f:
    prompts = [line.strip() for line in f if line.strip()]

for idx, prompt in enumerate(prompts, start=1):
    print(f"{idx:02d}: {prompt}")

# Generate images

One image per prompt and model. Each model is loaded, used and then released from GPU memory before the next one is loaded.

In [ ]:
def generate_images(model_id, output_dir):
    """Generate one image per prompt with the given model and save it to output_dir."""
    os.makedirs(output_dir, exist_ok=True)
    pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16).to("cuda")

    for idx, prompt in enumerate(prompts, start=1):
        print(f"Generating image {idx}/{len(prompts)}: {prompt}")
        generator = None
        if SEED is not None:
            generator = torch.Generator("cuda").manual_seed(SEED)
        image = pipe(prompt, generator=generator).images[0]
        image.save(os.path.join(output_dir, f"{idx:02d}_{prompt}.png"))

    # free GPU memory before the next model is loaded
    del pipe
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
assert torch.cuda.is_available(), "A CUDA GPU is required (on Colab: Runtime > Change runtime type > GPU)."

for version, model_id in MODELS.items():
    print(f"=== {version}: {model_id} ===")
    generate_images(model_id, os.path.join(OUTPUT_ROOT, version))